# DeepVision AI — CNN Training Notebook
## Goal
Train a Convolutional Neural Network to classify steel surface images into six defect categories, evaluate it honestly, and export the model for a local Streamlit app.

**Run in order:** use Google Colab, select a GPU if available, and run every cell from top to bottom. Training duration varies with runtime availability.


## 1. Install the dataset helper

In [ ]:
!pip -q install kagglehub


## 2. Download the public NEU dataset

In [ ]:
import os
from pathlib import Path
import kagglehub

download_path = Path(kagglehub.dataset_download("kaustubhdikshit/neu-surface-defect-database"))
print("Dataset downloaded to:", download_path)
for path in list(download_path.rglob("*"))[:40]:
    if path.is_dir():
        print("DIR ", path)


## 3. Find the train and validation image folders

In [ ]:
# The dataset commonly stores images as:
# NEU-DET/train/images/<class_name>/<image files>
# NEU-DET/validation/images/<class_name>/<image files>
# Find the 'images' folder under each split so Keras sees the class folders.

def find_class_folder_root(root, split_name):
    candidates = []
    for p in root.rglob("images"):
        if not p.is_dir():
            continue
        if p.parent.name.lower() != split_name.lower():
            continue
        class_dirs = [d for d in p.iterdir() if d.is_dir()]
        image_count = sum(
            1 for q in p.rglob("*")
            if q.is_file() and q.suffix.lower() in [".jpg", ".jpeg", ".png", ".bmp"]
        )
        if class_dirs and image_count:
            candidates.append((image_count, p))
    candidates.sort(key=lambda x: x[0], reverse=True)
    return candidates[0][1] if candidates else None

train_root = find_class_folder_root(download_path, "train")
test_root = find_class_folder_root(download_path, "validation")
if train_root is None or test_root is None:
    raise FileNotFoundError(
        "Could not automatically find train/validation class folders. "
        "Inspect the printed dataset folders above and update train_root/test_root."
    )

print("Train class-folder root:", train_root)
print("Held-out test class-folder root:", test_root)
print("Train image count:", sum(1 for p in train_root.rglob("*") if p.suffix.lower() in [".jpg", ".jpeg", ".png", ".bmp"]))
print("Test image count:", sum(1 for p in test_root.rglob("*") if p.suffix.lower() in [".jpg", ".jpeg", ".png", ".bmp"]))


## 4. Load libraries and inspect sample images

In [ ]:
import json, zipfile, shutil
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay
from pathlib import Path

SEED = 42
IMAGE_SIZE = (200, 200)
BATCH_SIZE = 32
tf.keras.utils.set_random_seed(SEED)

print("TensorFlow:", tf.__version__)
print("GPU devices:", tf.config.list_physical_devices("GPU"))

train_ds = tf.keras.utils.image_dataset_from_directory(
    train_root, labels="inferred", label_mode="int",
    image_size=IMAGE_SIZE, color_mode="grayscale",
    batch_size=BATCH_SIZE, validation_split=0.2, subset="training",
    seed=SEED, shuffle=True
)
val_ds = tf.keras.utils.image_dataset_from_directory(
    train_root, labels="inferred", label_mode="int",
    image_size=IMAGE_SIZE, color_mode="grayscale",
    batch_size=BATCH_SIZE, validation_split=0.2, subset="validation",
    seed=SEED, shuffle=False
)
test_ds = tf.keras.utils.image_dataset_from_directory(
    test_root, labels="inferred", label_mode="int",
    image_size=IMAGE_SIZE, color_mode="grayscale",
    batch_size=BATCH_SIZE, shuffle=False
)

class_names = train_ds.class_names
print("Class order:", class_names)
print("Classes:", len(class_names))

plt.figure(figsize=(10, 6))
for images, labels in train_ds.take(1):
    for i in range(min(12, len(images))):
        ax = plt.subplot(3, 4, i + 1)
        plt.imshow(tf.squeeze(images[i]), cmap="gray")
        plt.title(class_names[int(labels[i])])
        plt.axis("off")
plt.tight_layout()
plt.show()


## 5. Build a beginner-friendly CNN

In [ ]:
# Rescaling converts pixel values from 0–255 to 0–1.
# Convolution layers learn image patterns; pooling reduces spatial size.
# Dropout helps reduce overfitting.

AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.cache().prefetch(AUTOTUNE)
val_ds = val_ds.cache().prefetch(AUTOTUNE)
test_ds = test_ds.cache().prefetch(AUTOTUNE)

model = keras.Sequential([
    layers.Input(shape=(200, 200, 1)),
    layers.Rescaling(1./255),
    layers.Conv2D(16, 3, padding="same", activation="relu"),
    layers.MaxPooling2D(),
    layers.Conv2D(32, 3, padding="same", activation="relu"),
    layers.MaxPooling2D(),
    layers.Conv2D(64, 3, padding="same", activation="relu"),
    layers.MaxPooling2D(),
    layers.Dropout(0.25),
    layers.Flatten(),
    layers.Dense(64, activation="relu"),
    layers.Dropout(0.30),
    layers.Dense(len(class_names), activation="softmax")
])

model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.001),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)
model.summary()


## 6. Train the model

In [ ]:
callbacks = [
    keras.callbacks.EarlyStopping(
        monitor="val_loss", patience=5, restore_best_weights=True
    )
]
history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=20,
    callbacks=callbacks
)


## 7. Plot training curves

In [ ]:
plt.figure(figsize=(12, 4))
plt.subplot(1, 2, 1)
plt.plot(history.history["accuracy"], label="Train")
plt.plot(history.history["val_accuracy"], label="Validation")
plt.title("Accuracy by epoch")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(history.history["loss"], label="Train")
plt.plot(history.history["val_loss"], label="Validation")
plt.title("Loss by epoch")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()
plt.tight_layout()
plt.show()


## 8. Evaluate once on the held-out test set

In [ ]:
test_loss, test_accuracy = model.evaluate(test_ds, verbose=0)
print(f"Test loss: {test_loss:.4f}")
print(f"Test accuracy: {test_accuracy:.4f}")

y_true = np.concatenate([labels.numpy() for _, labels in test_ds])
y_prob = model.predict(test_ds, verbose=0)
y_pred = np.argmax(y_prob, axis=1)

print("\nClassification report:")
report_dict = classification_report(
    y_true, y_pred, target_names=class_names, output_dict=True, zero_division=0
)
print(classification_report(y_true, y_pred, target_names=class_names, zero_division=0))

cm = confusion_matrix(y_true, y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=class_names)
fig, ax = plt.subplots(figsize=(8, 8))
disp.plot(ax=ax, cmap="Blues", xticks_rotation=45, colorbar=False)
plt.title("Held-out test confusion matrix")
plt.tight_layout()
plt.show()


## 9. Export the trained model for the laptop app

In [ ]:
export_dir = Path("/content/deepvision_export")
export_dir.mkdir(parents=True, exist_ok=True)

model_path = export_dir / "deepvision_cnn.keras"
model.save(model_path)

with open(export_dir / "class_names.json", "w", encoding="utf-8") as f:
    json.dump(class_names, f, indent=2)

evaluation_summary = {
    "test_loss": float(test_loss),
    "test_accuracy": float(test_accuracy),
    "class_names": class_names,
    "note": "Metrics measured on the dataset's held-out validation split. This is a learning benchmark, not a guarantee of real-world industrial performance."
}
with open(export_dir / "evaluation_summary.json", "w", encoding="utf-8") as f:
    json.dump(evaluation_summary, f, indent=2)

zip_path = "/content/deepvision_export.zip"
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
    for file_path in export_dir.iterdir():
        zf.write(file_path, arcname=file_path.name)

print("Export ready:", zip_path)
print(json.dumps(evaluation_summary, indent=2))

from google.colab import files
files.download(zip_path)


## 10. What you should understand before the interview

- Convolution learns local visual features; pooling reduces feature-map dimensions.
- ReLU adds non-linearity; softmax produces scores across the six classes.
- Cross-entropy measures classification loss; Adam updates weights using gradients.
- Training accuracy alone is not enough: inspect validation curves and held-out test metrics.
- The model predicts one label for the entire image. It does not mark the exact defect location.
- If train accuracy rises while validation performance stalls or worsens, that is a sign of overfitting.
